In [224]:
# import libraries and api key for FRED
import json
import time
import requests
from tqdm import tqdm
import os
import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from bs4 import BeautifulSoup as bs

load_dotenv()
fred = Fred(api_key=os.environ["FRED_API_KEY"])

In [180]:
# get the time series data from FRED and Yahoo Finance

icsa = fred.get_series("ICSA", observation_start='1995-01-01')
dgs2 = fred.get_series("DGS2", observation_start='1995-01-01')
dgs10 = fred.get_series("DGS10", observation_start='1995-01-01')
cpi = fred.get_series('CPIAUCSL', observation_start='1995-01-01')
vix = yf.download('^VIX', start='1995-01-01')['Close'].squeeze()

[*********************100%***********************]  1 of 1 completed


In [181]:
print(icsa.head(), dgs2.head(), dgs10.head(), vix.head())

1995-01-07    338000.0
1995-01-14    347000.0
1995-01-21    325000.0
1995-01-28    324000.0
1995-02-04    324000.0
dtype: float64 1995-01-02     NaN
1995-01-03    7.73
1995-01-04    7.62
1995-01-05    7.66
1995-01-06    7.64
dtype: float64 1995-01-02     NaN
1995-01-03    7.88
1995-01-04    7.82
1995-01-05    7.88
1995-01-06    7.87
dtype: float64 Date
1995-01-03    14.25
1995-01-04    13.53
1995-01-05    13.50
1995-01-06    13.13
1995-01-09    13.33
Name: ^VIX, dtype: float64


In [182]:
# create a date range index from 1996-01-01 to today with business day frequency, calculate yield spread, yearly inflation, and reindex all series on new index

idx = pd.date_range(start='1996-01-01', end='today', freq='B')

vix = vix.reindex(idx, method='ffill')

dgs2.index = pd.to_datetime(dgs2.index)
dgs10.index = pd.to_datetime(dgs10.index)
spread = (dgs10 - dgs2).reindex(idx, method='ffill')

icsa.index = pd.to_datetime(icsa.index)
icsa = icsa.reindex(idx, method='ffill')

cpi.index = pd.to_datetime(cpi.index)
cpiYoY = cpi.pct_change(12) * 100
cpiYoY = cpiYoY.reindex(idx, method='ffill')

df = pd.DataFrame({
    'icsa': icsa,
    'spread': spread,
    'vix': vix,
    'cpiYoY': cpiYoY
})

print(df.head())
print(df.tail())

                icsa  spread    vix    cpiYoY
1996-01-01  359000.0     NaN  12.52  2.790698
1996-01-02  359000.0    0.42  12.19  2.790698
1996-01-03  359000.0    0.41  12.10  2.790698
1996-01-04  359000.0    0.48  13.78  2.790698
1996-01-05  359000.0    0.49  13.58  2.790698
                icsa  spread        vix    cpiYoY
2026-06-29  215000.0    0.28  17.650000  4.166615
2026-06-30  215000.0    0.30  16.450001  4.166615
2026-07-01  215000.0    0.31  16.590000  4.166615
2026-07-02  215000.0    0.31  16.150000  4.166615
2026-07-03  215000.0    0.31  15.800000  4.166615


In [183]:
# calculate z-scores for each series, using 1260 day window for inflation, drop any rows with NaN values, and convert to numpy array

df_z = pd.DataFrame(index=df.index)

windows = {
    'icsa': 252,
    'spread': 252,
    'vix': 252,
    'cpiYoY': 1260
}

for col, window in windows.items():
    min_periods = int(window * 0.9)
    rollingMean = df[col].rolling(window, min_periods=min_periods).mean()
    rollingStd = df[col].rolling(window, min_periods=min_periods).std()
    df_z[col] = (df[col] - rollingMean) / rollingStd


df_z = df_z.dropna()
df = df.dropna()

print(df_z)
z = df_z.values
print(z)

x = df.values

                icsa    spread       vix    cpiYoY
2000-05-04  0.312858 -1.692789  2.825877  1.304183
2000-05-05  0.316941 -1.756334  1.541560  1.302632
2000-05-08  0.487929 -1.615050  1.787052  1.301087
2000-05-09  0.491674 -1.719857  2.047216  1.299548
2000-05-10  0.493921 -1.861623  2.390375  1.298013
...              ...       ...       ...       ...
2026-06-29 -0.346967 -3.021542 -0.142337 -0.157835
2026-06-30 -0.342910 -2.743011 -0.502332 -0.157467
2026-07-01 -0.340832 -2.586108 -0.459490 -0.157029
2026-07-02 -0.338757 -2.540462 -0.589614 -0.156591
2026-07-03 -0.336683 -2.496116 -0.692335 -0.156153

[6522 rows x 4 columns]
[[ 0.31285834 -1.69278852  2.82587682  1.30418312]
 [ 0.31694073 -1.75633437  1.54155962  1.30263242]
 [ 0.48792854 -1.61505011  1.78705176  1.30108725]
 ...
 [-0.34083205 -2.5861083  -0.45949035 -0.15702945]
 [-0.33875651 -2.54046229 -0.58961412 -0.15659143]
 [-0.33668307 -2.49611626 -0.69233505 -0.1561534 ]]


In [184]:
# fit a Gaussian HMM to the z-scores and calculate the BIC for each number of components from 2 to 6, then print the BIC scores and the optimal number of components

bic_scores = {}
for k in range(2, 6):
    model = GaussianHMM(n_components=k, covariance_type='full', n_iter=1000, random_state=42)
    model.fit(z)
    bic_scores[k] = model.bic(z)

print(bic_scores)
best_k = min(bic_scores, key=bic_scores.get)
print(best_k)

{2: np.float64(78559.24009765284), 3: np.float64(81111.16729785626), 4: np.float64(68480.52499106144), 5: np.float64(71656.50162074463)}
4


In [185]:
# fit the HMM with the opttimal no. components, predict hidden states, and add to dataframe, test how many days in each state

model = GaussianHMM(n_components=best_k, covariance_type='full', n_iter=1000, random_state=42)
model.fit(z)

states = model.predict(z)
df_z['state'] = states

print(df_z.head())
df_z['state'].value_counts()

                icsa    spread       vix    cpiYoY  state
2000-05-04  0.312858 -1.692789  2.825877  1.304183      1
2000-05-05  0.316941 -1.756334  1.541560  1.302632      1
2000-05-08  0.487929 -1.615050  1.787052  1.301087      1
2000-05-09  0.491674 -1.719857  2.047216  1.299548      1
2000-05-10  0.493921 -1.861623  2.390375  1.298013      1


state
2    1965
3    1650
0    1468
1    1439
Name: count, dtype: int64

In [186]:
# Print means for each state
print(pd.DataFrame(model.means_, columns=['icsa', 'spread', 'vix', 'cpiYoY']))

df_z['year'] = df_z.index.year
df_z['state'] = states

# Create a pivot table showing the number of days in each state for each year
pivot = df_z.groupby(['year', 'state']).size().unstack(fill_value=0)
print(pivot)

# Have a look at what the model says about the last month
print(df_z.tail())

       icsa    spread       vix    cpiYoY
0  1.313541  1.421943  0.610603 -0.041955
1 -0.960735 -0.990627  0.687921  1.264229
2 -0.398713 -1.276020 -0.767927  0.097029
3 -0.812212  1.141934 -0.523030 -0.500754
state    0    1    2    3
year                     
2000    62   26   78    0
2001   233    0    0   15
2002    45   29    0  176
2003    90    0   27  133
2004     0   19  188   43
2005    24   62  164    0
2006     9  105  136    0
2007   150   16   38   47
2008   251    0    0    0
2009   188    0    0   62
2010     0   58   85  108
2011    10  144   14   82
2012     4    3  233   10
2013    26    0    0  224
2014     9   54  131   56
2015     0   67  141   43
2016     9   50  165   26
2017     4  104  142    0
2018     1  216   32    0
2019    64   26  138   22
2020    91   22    0  138
2021    23  190    0   38
2022     0  215   34    0
2023    14    0  167   69
2024   107    0    0  143
2025    54    0    0  173
2026     0   33   52   42
                icsa    spread      

In [187]:
# Add state labels to df
states = {
    0: 'Recession/Stress',
    1: 'Inflation/Tightening',
    2: 'Late Cycle/Complacency',
    3: 'Early Recovery/Expansiom'
}

df_z['state_label'] = df_z['state'].map(states)
print(df_z.tail())

                icsa    spread       vix    cpiYoY  state  year  \
2026-06-29 -0.346967 -3.021542 -0.142337 -0.157835      1  2026   
2026-06-30 -0.342910 -2.743011 -0.502332 -0.157467      2  2026   
2026-07-01 -0.340832 -2.586108 -0.459490 -0.157029      2  2026   
2026-07-02 -0.338757 -2.540462 -0.589614 -0.156591      2  2026   
2026-07-03 -0.336683 -2.496116 -0.692335 -0.156153      2  2026   

                       state_label  
2026-06-29    Inflation/Tightening  
2026-06-30  Late Cycle/Complacency  
2026-07-01  Late Cycle/Complacency  
2026-07-02  Late Cycle/Complacency  
2026-07-03  Late Cycle/Complacency  


In [228]:
# FOMC minutes index, and find minutes links, filter to html only

url = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"
response = requests.get(url)
soup = bs(response.content, 'html.parser')

links = soup.find_all('a', href=True)
minutes_links = [l['href'] for l in links if 'minutes' in l['href'].lower()]
print(f"Found {len(minutes_links)} minutes links")
print(minutes_links[:5])

minutes_links_htm = [l for l in minutes_links if l.endswith('.htm')]
print(f"HTM links: {len(minutes_links_htm)}")
print(minutes_links_htm)

Found 86 minutes links
['/monetarypolicy/files/fomcminutes20260128.pdf', '/monetarypolicy/fomcminutes20260128.htm', '/monetarypolicy/files/fomcminutes20260318.pdf', '/monetarypolicy/fomcminutes20260318.htm', '/monetarypolicy/files/fomcminutes20260429.pdf']
HTM links: 43
['/monetarypolicy/fomcminutes20260128.htm', '/monetarypolicy/fomcminutes20260318.htm', '/monetarypolicy/fomcminutes20260429.htm', '/monetarypolicy/fomcminutes20250129.htm', '/monetarypolicy/fomcminutes20250319.htm', '/monetarypolicy/fomcminutes20250507.htm', '/monetarypolicy/fomcminutes20250618.htm', '/monetarypolicy/fomcminutes20250730.htm', '/monetarypolicy/fomcminutes20250917.htm', '/monetarypolicy/fomcminutes20251029.htm', '/monetarypolicy/fomcminutes20251210.htm', '/monetarypolicy/fomcminutes20240131.htm', '/monetarypolicy/fomcminutes20240320.htm', '/monetarypolicy/fomcminutes20240501.htm', '/monetarypolicy/fomcminutes20240612.htm', '/monetarypolicy/fomcminutes20240731.htm', '/monetarypolicy/fomcminutes20240918.htm

In [ ]:
# extract the text and dates. also look at the text in the middle, as thats where the real discussion is

base_url = "https://www.federalreserve.gov"

def get_minutes_text(path):
    url = base_url + path
    response = requests.get(url)
    soup = bs(response.content, 'html.parser')
    content = soup.find('div', {'id': 'content'}) or soup.find('article')
    if content:
        return content.get_text(separator=' ', strip=True)
    return soup.get_text(separator=' ', strip=True)

def extract_date(path):
    import re
    match = re.search(r'(\d{8})\.htm', path)
    if match:
        return match.group(1)
    return None

test_text = get_minutes_text(minutes_links_htm[3])  # try a different one
# print(test_text[:3000])
test_date = extract_date(minutes_links_htm[3])
mid = len(test_text) // 3
print(test_text[mid:mid+1000])
# print(f"Date: {test_date}")
# print(f"Length: {len(test_text)} chars")
# print(f"Preview: {test_text[:500]}")

od prices. By contrast, in some Latin American countries, most notably Brazil, inflation continued to increase, in part spurred by currency depreciation. Many foreign central banks eased policy during the intermeeting period, including the Bank of Canada, the European Central Bank, and the Swiss National Bank among the AFEs, and the central banks of Colombia, Indonesia, and Mexico among the emerging market economies (EMEs). In contrast, amid improving economic conditions, the Bank of Japan increased its policy rate, taking a further step in gradually removing policy accommodation. Staff Review of the Financial Situation The market-implied path of the federal funds rate over the next year was little changed on net over the intermeeting period. Yields on shorter-term nominal Treasury securities edged down, but longer-term rates rose moderately. The increase in longer-term yields appeared to be partly attributable to higher term premiums. Measures of inflation compensation over the near t

In [248]:
# extract text from all and classify using groq, start where substative content starts to remove the start waffle

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

def classify_minutes(text):
    # target policy decision section rather than staff economic review
    markers = [
        'Committee Policy Action',
        'Participants\' Views on Current Conditions',
        'In their discussion of monetary policy',
        'Members agreed'
    ]
    
    start = None
    for marker in markers:
        idx = text.find(marker)
        if idx != -1:
            start = idx
            break
    
    if start is None:
        idx = text.find('Staff Review of the Economic Situation')
        start = idx if idx != -1 else len(text) // 4

    truncated = text[start:start+4000]
    
    prompt = f"""You are a financial analyst reading Federal Reserve FOMC minutes.
Focus specifically on the US economic outlook and Federal Reserve policy stance.
Classify the prevailing tone and narrative by scoring the following four themes from 0.0 to 1.0. Scores must sum to 1.0.
Distribute scores across all four themes based on what you actually read in the text.

Themes:
- crisis_recession_stress: US recession fears, financial instability, emergency Fed policy, economic contraction risk
- inflation_tightening: US inflation concerns, Fed rate hikes, hawkish stance, tightening monetary policy
- latecycle_complacency: Fed holding rates steady, higher for longer narrative, soft landing confidence, monitoring stance
- early_recovery_expansion: US growth optimism, Fed easing, rate cuts, strong labour market, expansion confidence

Text:
{truncated}

Read the text carefully and assign scores reflecting what the text actually discusses.
Respond only in JSON with no preamble or markdown using these exact keys:
crisis_recession_stress, inflation_tightening, latecycle_complacency, early_recovery_expansion"""

    response = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers={
            'Authorization': f'Bearer {GROQ_API_KEY}',
            'Content-Type': 'application/json'
        },
        json={
            'model': 'llama-3.1-8b-instant',
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0.3
        }
    )
    
    data = response.json()
    if 'error' in data:
        raise ValueError(f"API error: {data['error'].get('message', 'unknown')}")
    
    content = data['choices'][0]['message']['content']
    return json.loads(content)

test_paths = [
    '/monetarypolicy/fomcminutes20210127.htm',
    '/monetarypolicy/fomcminutes20210428.htm',
    '/monetarypolicy/fomcminutes20211215.htm'
]

for path in test_paths:
    date = extract_date(path)
    text = get_minutes_text(path)
    result = classify_minutes(text)
    print(f"{date}: {result}")
    time.sleep(8)

20210127: {'crisis_recession_stress': 0.6, 'inflation_tightening': 0.1, 'latecycle_complacency': 0.2, 'early_recovery_expansion': 0.1}
20210428: {'crisis_recession_stress': 0.6, 'inflation_tightening': 0.2, 'latecycle_complacency': 0.1, 'early_recovery_expansion': 0.1}
20211215: {'crisis_recession_stress': 0.2, 'inflation_tightening': 0.5, 'latecycle_complacency': 0.2, 'early_recovery_expansion': 0.1}


In [251]:
fomc_results = []

for path in tqdm(minutes_links_htm):
    date = extract_date(path)
    if not date:
        continue
    
    try:
        text = get_minutes_text(path)
        scores = classify_minutes(text)
        total = sum(v for v in scores.values() if v is not None)
        if total == 0:
            print(f"Zero scores for {date}")
            continue
        scores = {k: v/total for k, v in scores.items()}
        scores['date'] = date
        fomc_results.append(scores)
        print(f"{date}: {scores}")
        time.sleep(10)
        
    except Exception as e:
        print(f"Failed on {date}: {e}")

fomc_sentiment = pd.DataFrame(fomc_results)
fomc_sentiment.to_csv('fomc_sentiment.csv', index=False)
print(f"\nTotal: {len(fomc_sentiment)} minutes classified")
print(fomc_sentiment)

  0%|          | 0/43 [00:00<?, ?it/s]

20260128: {'crisis_recession_stress': 0.1, 'inflation_tightening': 0.7, 'latecycle_complacency': 0.1, 'early_recovery_expansion': 0.1, 'date': '20260128'}


  2%|▏         | 1/43 [00:10<07:20, 10.48s/it]

20260318: {'crisis_recession_stress': 0.2, 'inflation_tightening': 0.5, 'latecycle_complacency': 0.2, 'early_recovery_expansion': 0.1, 'date': '20260318'}


  5%|▍         | 2/43 [00:20<07:09, 10.47s/it]

...

20211103: {'crisis_recession_stress': 0.2, 'inflation_tightening': 0.4, 'latecycle_complacency': 0.2, 'early_recovery_expansion': 0.2, 'date': '20211103'}


 98%|█████████▊| 42/43 [07:21<00:10, 10.54s/it]

20211215: {'crisis_recession_stress': 0.3, 'inflation_tightening': 0.5, 'latecycle_complacency': 0.1, 'early_recovery_expansion': 0.1, 'date': '20211215'}


100%|██████████| 43/43 [07:32<00:00, 10.52s/it]


Total: 43 minutes classified
    crisis_recession_stress  inflation_tightening  latecycle_complacency  \
0                      0.10                  0.70                   0.10   
1                      0.20                  0.50                   0.20   
2                      0.20                  0.50                   0.20   
3                      0.20                  0.40                   0.20   
4                      0.20                  0.40                   0.20   
5                      0.20                  0.50                   0.10   
6                      0.05                  0.45                   0.30   
7                      0.40                  0.30                   0.20   
8                      0.45                  0.35                   0.10   
9                      0.25                  0.40                   0.20   
10                     0.25                  0.40                   0.20   
11                     0.20                  0.60         

In [275]:
# load csv of fed sentiment and reindex to match df_z, and forward fill to be daily (as only ~8 discussions per year)

fomc = pd.read_csv('fomc_sentiment.csv')
fomc['date'] = pd.to_datetime(fomc['date'], format='%Y%m%d')
fomc = fomc.set_index('date').sort_index()

fomc_daily = fomc.reindex(df_z.index, method='ffill')

combined_fomc = df_z[['state']].join(fomc_daily, how='inner')
combined_fomc = combined_fomc.dropna()

print(combined_fomc.head())
print(combined_fomc.shape)
combined_fomc.to_csv('combined_fomc.csv')

            state  crisis_recession_stress  inflation_tightening  \
2021-01-27      3                      0.5                   0.2   
2021-01-28      3                      0.5                   0.2   
2021-01-29      3                      0.5                   0.2   
2021-02-01      3                      0.5                   0.2   
2021-02-02      3                      0.5                   0.2   

            latecycle_complacency  early_recovery_expansion  
2021-01-27                   0.15                      0.15  
2021-01-28                   0.15                      0.15  
2021-01-29                   0.15                      0.15  
2021-02-01                   0.15                      0.15  
2021-02-02                   0.15                      0.15  
(1338, 5)


In [276]:
# identify market regime transition dates
combined_fomc['state_change'] = combined_fomc['state'] != combined_fomc['state'].shift(1)
transitions = combined_fomc[combined_fomc['state_change']].index
print(f"Total transitions: {len(transitions)}")
print(transitions)

# filter out transitions less than 30 days apart
transitions_filtered = [transitions[0]]
for t in transitions[1:]:
    if (t - transitions_filtered[-1]).days >= 30:
        transitions_filtered.append(t)

transitions_filtered = pd.DatetimeIndex(transitions_filtered)
print(f"Filtered transitions: {len(transitions_filtered)}")
print(transitions_filtered)

Total transitions: 23
DatetimeIndex(['2021-01-27', '2021-03-01', '2021-04-01', '2022-11-10',
               '2023-06-05', '2023-06-26', '2023-09-21', '2024-06-06',
               '2024-11-06', '2024-12-18', '2024-12-20', '2025-02-24',
               '2025-03-14', '2025-04-03', '2025-05-05', '2025-06-02',
               '2025-06-23', '2025-09-08', '2025-09-15', '2026-03-05',
               '2026-04-09', '2026-06-17', '2026-06-30'],
              dtype='datetime64[us]', freq=None)
Filtered transitions: 17
DatetimeIndex(['2021-01-27', '2021-03-01', '2021-04-01', '2022-11-10',
               '2023-06-05', '2023-09-21', '2024-06-06', '2024-11-06',
               '2024-12-18', '2025-02-24', '2025-04-03', '2025-05-05',
               '2025-06-23', '2025-09-08', '2026-03-05', '2026-04-09',
               '2026-06-17'],
              dtype='datetime64[us]', freq=None)


In [282]:
sentiment_cols = ['crisis_recession_stress', 'inflation_tightening',
                  'latecycle_complacency', 'early_recovery_expansion']

results_score = []

theme_map = {
    0: 'crisis_recession_stress',
    1: 'inflation_tightening',
    2: 'latecycle_complacency',
    3: 'early_recovery_expansion'
}

for transition_date in transitions_filtered:
    new_state = combined_fomc.loc[transition_date, 'state']
    target_theme = theme_map.get(new_state)
    if not target_theme:
        continue
    
    # only look at last 8 FOMC meetings before transition
    fomc_before = fomc[fomc.index < transition_date].sort_index().tail(8)
    
    if len(fomc_before) < 3:
        continue
    
    theme_scores = fomc_before[target_theme]
    
    # find first meeting in this local window where score was rising
    # i.e. higher than the meeting before it
    rising_dates = []
    for i in range(1, len(theme_scores)):
        if theme_scores.iloc[i] > theme_scores.iloc[i-1]:
            rising_dates.append(theme_scores.index[i])
    
    if rising_dates:
        first_rise_date = rising_dates[0]
        days_lead = (transition_date - first_rise_date).days
    else:
        first_rise_date = None
        days_lead = None
    
    results_score.append({
        'transition_date': transition_date,
        'new_state': new_state,
        'target_theme': target_theme,
        'first_rise_date': first_rise_date,
        'days_lead': days_lead,
        'scores': theme_scores.values.tolist()
    })

results_score_df = pd.DataFrame(results_score)
print(results_score_df[['transition_date', 'new_state', 'target_theme', 
                         'first_rise_date', 'days_lead']].to_string())
print(f"\nMean lead days: {results_score_df['days_lead'].dropna().mean():.0f}")
print(f"Median lead days: {results_score_df['days_lead'].dropna().median():.0f}")

   transition_date  new_state              target_theme first_rise_date  days_lead
0       2022-11-10          2     latecycle_complacency      2022-01-26        288
1       2023-06-05          0   crisis_recession_stress      2023-03-22         75
2       2023-09-21          3  early_recovery_expansion      2022-12-14        281
3       2024-06-06          0   crisis_recession_stress      2023-11-01        218
4       2024-11-06          3  early_recovery_expansion      2024-01-31        280
5       2024-12-18          0   crisis_recession_stress      2024-09-18         91
6       2025-02-24          0   crisis_recession_stress      2024-09-18        159
7       2025-04-03          0   crisis_recession_stress      2024-09-18        197
8       2025-05-05          3  early_recovery_expansion      2024-12-18        138
9       2025-06-23          3  early_recovery_expansion      2024-12-18        187
10      2025-09-08          0   crisis_recession_stress      2025-01-29        222
11  

In [280]:
# December 2021 pivot case study
dec_2021 = fomc.loc['2021-12-15']
nov_2021 = fomc.loc['2021-11-03']
print("November 2021 dominant:", nov_2021[sentiment_cols].idxmax())
print("December 2021 dominant:", dec_2021[sentiment_cols].idxmax())
print("Next HMM transition:", transitions_filtered[transitions_filtered > pd.Timestamp('2021-12-15')][0])

print(fomc.loc['2021-07-28':'2021-12-15'][sentiment_cols])

November 2021 dominant: inflation_tightening
December 2021 dominant: inflation_tightening
Next HMM transition: 2022-11-10 00:00:00
            crisis_recession_stress  inflation_tightening  \
date                                                        
2021-07-28                     0.40                   0.3   
2021-09-22                     0.25                   0.4   
2021-11-03                     0.20                   0.4   
2021-12-15                     0.30                   0.5   

            latecycle_complacency  early_recovery_expansion  
date                                                         
2021-07-28                   0.20                       0.1  
2021-09-22                   0.25                       0.1  
2021-11-03                   0.20                       0.2  
2021-12-15                   0.10                       0.1  
